# 03 — Optimizer types: how SkillOpt accepts an edit

The optimizer takes a merged patch (from `gradient.aggregate`) and decides how to apply it. SkillOpt ships several strategies:

| Strategy | Module | What it does |
|---|---|---|
| **clip** | `optimizer/clip.py` | Limit number of edits per step (textual learning rate budget) |
| **slow_update** | `optimizer/slow_update.py` | Rewrite the skill in measured passes; accept if score improves |
| **meta_skill** | `optimizer/meta_skill.py` | Maintain an extra "meta" skill document distilled across epochs |
| **rewrite** | `optimizer/rewrite.py` | Wholesale rewrite of the skill from current state + patch |
| **select** | `optimizer/select.py` | Pick the best subset of proposed edits |

This notebook reads each one in turn. No LLM calls.

In [1]:
import inspect, os
from skillopt import optimizer

opt_root = os.path.dirname(optimizer.__file__)
print("optimizer/ at:", opt_root)
print()
for f in sorted(os.listdir(opt_root)):
    if f.endswith(".py") and f != "__init__.py":
        path = os.path.join(opt_root, f)
        lines = sum(1 for _ in open(path))
        print(f"  {f:<30} {lines:>5} lines")

optimizer/ at: /Users/mhuang/Documents/GitHub/SkillOpt/skillopt/optimizer

  clip.py                          109 lines
  lr_autonomous.py                 108 lines
  meta_skill.py                     87 lines
  rewrite.py                        59 lines
  scheduler.py                     127 lines
  select.py                          4 lines
  skill.py                         164 lines
  slow_update.py                   393 lines
  update_modes.py                  136 lines


## 1. `clip` — the textual learning rate

The "edit budget" L caps how many edits get applied per step. Look at `apply_clip`:

In [2]:
from skillopt.optimizer import clip as clip_mod

public = [
    n for n in dir(clip_mod) if not n.startswith("_") and callable(getattr(clip_mod, n))
]
print("clip.py callables:", public)
print()
src = inspect.getsource(clip_mod)
print(src[:2200] + ("\n..." if len(src) > 2200 else ""))

clip.py callables: ['chat_optimizer', 'describe_item', 'extract_json', 'format_meta_skill_context', 'get_payload_items', 'is_rew
rite_mode', 'load_prompt', 'normalize_update_mode', 'payload_key', 'payload_label', 'rank_and_select']

"""ReflACT gradient clipping — LLM-driven edit ranking and selection.

Analogous to gradient clipping in neural network training: ranks candidate
edits by importance and selects the top-L to apply, controlling the
effective step size. Previously core/select.py.
"""
from __future__ import annotations

from skillopt.model import chat_optimizer
from skillopt.optimizer.meta_skill import format_meta_skill_context
from skillopt.optimizer.update_modes import (
    describe_item,
    get_payload_items,
    is_rewrite_mode,
    normalize_update_mode,
    payload_key,
    payload_label,
)
from skillopt.prompts import load_prompt
from skillopt.utils import extract_json


# ── Public API ────────────────────────────────────────────────────────────────

def rank_and_sel

## 2. `select` — rank and pick the top-K edits

In [3]:
from skillopt.optimizer import select as select_mod

public = [
    n
    for n in dir(select_mod)
    if not n.startswith("_") and callable(getattr(select_mod, n))
]
print("select.py callables:", public)
print()

# Show the entry function
for name in ("rank_and_select", "select_edits", "select"):
    fn = getattr(select_mod, name, None)
    if fn and inspect.isfunction(fn):
        print(f"--- {name}{inspect.signature(fn)} ---")
        src = inspect.getsource(fn)
        print(src[:1800] + ("\n..." if len(src) > 1800 else ""))
        break

select.py callables: ['rank_and_select']

--- rank_and_select(skill_content: 'str', patch: 'dict', max_edits: 'int', meta_skill_context: 'str' = '', update_mode: 'str' =
'patch') -> 'dict' ---
def rank_and_select(
    skill_content: str,
    patch: dict,
    max_edits: int,
    meta_skill_context: str = "",
    update_mode: str = "patch",
) -> dict:
    """Use a optimizer LLM to rank edits by importance, then keep top-L.

    If the edit pool is within budget, returns the patch unchanged.
    Otherwise, calls the optimizer to rank and select the most impactful edits.

    Parameters
    ----------
    skill_content : str
        Current skill document.
    patch : dict
        Merged :class:`~skillopt.types.Patch` dict with ``edits`` list.
    max_edits : int
        Maximum number of edits to keep (the "edit budget").

    Returns
    -------
    dict
        :class:`~skillopt.types.Patch` dict with selected edits and
        optional ``ranking_details``.
    """
    update_mode = nor

## 3. `slow_update` — controlled rewrite

In [4]:
from skillopt.optimizer import slow_update as su_mod

print(
    "slow_update.py callables:",
    [n for n in dir(su_mod) if not n.startswith("_") and callable(getattr(su_mod, n))][
        :10
    ],
)
print()

# Find main function
for name in ("apply_slow_update", "slow_update", "run_slow_update"):
    fn = getattr(su_mod, name, None)
    if fn and inspect.isfunction(fn):
        print(f"--- {name}{inspect.signature(fn)} ---")
        src = inspect.getsource(fn)
        print(src[:1500] + ("\n..." if len(src) > 1500 else ""))
        break

slow_update.py callables: ['build_comparison_pairs', 'chat_optimizer', 'extract_json', 'extract_slow_update_field', 'format_comp
arison_text', 'has_slow_update_field', 'inject_empty_slow_update_field', 'load_prompt', 'replace_slow_update_field', 'run_slow_u
pdate']

--- run_slow_update(skill_content: 'str', results_prev: 'list[dict]', results_curr: 'list[dict]', items: 'list[dict]', *, prev_s
kill: 'str' = '', prev_slow_update_content: 'str' = '', prev_rollout_dir: 'str' = '', curr_rollout_dir: 'str' = '', comparison_p
airs: 'list[dict] | None' = None, system_prompt: 'str | None' = None) -> 'dict | None' ---
def run_slow_update(
    skill_content: str,
    results_prev: list[dict],
    results_curr: list[dict],
    items: list[dict],
    *,
    prev_skill: str = "",
    prev_slow_update_content: str = "",
    prev_rollout_dir: str = "",
    curr_rollout_dir: str = "",
    comparison_pairs: list[dict] | None = None,
    system_prompt: str | None = None,
) -> dict | None:
    """Run the 

## 4. `apply_patch` — the actual edit-to-text transformer

This is the deterministic part: given a `Patch` dataclass with edits, produce the new skill markdown. No LLM involved — just text surgery on the markdown.

In [5]:
from skillopt.optimizer import apply_patch as ap_mod

public = [
    n for n in dir(ap_mod) if not n.startswith("_") and callable(getattr(ap_mod, n))
]
print("apply_patch.py callables:", public)
print()

for name in ("apply_patch", "apply_edits"):
    fn = getattr(ap_mod, name, None)
    if fn and inspect.isfunction(fn):
        print(f"--- {name}{inspect.signature(fn)} ---")
        src = inspect.getsource(fn)
        print(src[:1800] + ("\n..." if len(src) > 1800 else ""))
        print()

apply_patch.py callables: []


## 5. Try `apply_patch` against a tiny skill — deterministic, no LLM

The text-surgery layer is pure Python. We can drive it directly to see the edit ops in action.

In [7]:
from skillopt.types import Edit, Patch

# apply_patch resolves directly to a function in the optimizer package
from skillopt.optimizer import apply_patch as apply_patch_fn

skill_before = """\
# Math solver skill

## Approach
1. Read the problem.
2. Solve.
"""

patch = Patch(
    reasoning="Add format requirement and verification step",
    edits=[
        Edit(op="append", content="\n## Format\nEnd with '### <answer>'.\n"),
        Edit(
            op="insert_after",
            target="2. Solve.",
            content="3. Verify the answer fits the constraints.\n",
        ),
    ],
)

# Inspect the function's signature first
print("apply_patch signature:", inspect.signature(apply_patch_fn))
print()

skill_after = apply_patch_fn(skill_before, patch.to_dict())

print("=== BEFORE ===")
print(skill_before)
print("=== PATCH ===")
print(patch.to_dict())
print()
print("=== AFTER ===")
print(skill_after)

apply_patch signature: (skill: 'str', patch: 'PatchType | dict') -> 'str'

=== BEFORE ===
# Math solver skill

## Approach
1. Read the problem.
2. Solve.

=== PATCH ===
{'reasoning': 'Add format requirement and verification step', 'edits': [{'op': 'append', 'content': "\n## Format\nEnd with '###
<answer>'.\n"}, {'op': 'insert_after', 'content': '3. Verify the answer fits the constraints.\n', 'target': '2. Solve.'}]}

=== AFTER ===
# Math solver skill

## Approach
1. Read the problem.
2. Solve.

3. Verify the answer fits the constraints.

## Format
End with '### <answer>'.


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. `clip` — the textual learning rate
- 2. `select` — rank and pick the top-K edits
- 3. `slow_update` — controlled rewrite
- 4. `apply_patch` — the actual edit-to-text transformer
- 5. Try `apply_patch` against a tiny skill — deterministic, no LLM

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import json as _json
from pathlib import Path as _Path
import collections as _c

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/03-optimizer-types.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print(f"output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")


## Data sources

| Source | Path |
|---|---|
| `optimizer.clip` | `skillopt/optimizer/clip.py` |
| `optimizer.select.rank_and_select` | `skillopt/optimizer/select.py` |
| `optimizer.slow_update` | `skillopt/optimizer/slow_update.py` |
| `optimizer.apply_patch` | `skillopt/optimizer/apply_patch.py` |
| `Edit` / `Patch` types | `skillopt/types.py` |
| Example skill text + patch | constructed in this notebook (minimal fixture exercising the pure text-surgery path; no LLM involvement) |

→ **Next:** [`04-scheduler-and-budget.ipynb`](04-scheduler-and-budget.ipynb).